Lecture 2 focuses on **Tabular MDP Planning**, exploring how to evaluate and find optimal policies when the environment's dynamics and rewards are fully known.

### **Core Definitions and Concepts**
*   **Markov Decision Process (MDP):** Defined by a tuple ($S, A, P, R, \gamma$), where $S$ is the state space, $A$ is the action space, $P$ is the dynamics model, $R$ is the reward model, and $\gamma$ is the discount factor. 
*   **Discount Factor ($\gamma$):** A value between 0 and 1 that weights future rewards. A large $\gamma$ (close to 1) means the agent cares significantly about long-term rewards, while $\gamma=0$ makes the agent "myopic," caring only about immediate rewards. A $\gamma < 1$ ensures that the infinite sum of rewards remains a **finite number**.
*   **Policy ($\pi$):** A mapping from states to actions, which can be **deterministic** (one action per state) or **stochastic** (a probability distribution over actions). In a tabular, infinite-horizon MDP, there is always at least one stationary and deterministic optimal policy.
*   **Q-Function (State-Action Value Function):** The expected discounted sum of future rewards starting from state $s$, taking action $a$, and then following policy $\pi$ thereafter.

### **Policy Evaluation**
Policy evaluation is the process of determining the **Value Function** ($V^\pi$), which represents the average reward an agent expects to receive starting in a specific state and following a fixed policy forever.
*   **Analytical Solution:** $V$ can be solved directly using a matrix equation: $V = (I - \gamma P)^{-1} R$. This requires a matrix inversion, which is computationally expensive ($O(n^3)$) and only feasible for small state spaces.
*   **Iterative Policy Evaluation:** This uses **dynamic programming** to repeatedly apply the **Bellman backup** for a policy until the value function stops changing.
*   **Simulation:** For very large state spaces, an agent can "roll out" episodes and average the rewards to estimate the value without knowing the explicit dynamics model.

### **Algorithms for Optimal Planning**
The lecture compares two primary algorithms for finding the optimal policy ($\pi^*$) and value function ($V^*$):

*   **Policy Iteration:**
    *   **Process:** It alternates between **policy evaluation** (computing $V^\pi$ for the current policy) and **policy improvement** (extracting a new policy by taking the **argmax** of the Q-function).
    *   **Property:** It is guaranteed to show **monotonic improvement**, meaning every new policy is strictly better than the last for every state until the optimal policy is reached.
    *   **Termination:** It stops when the policy no longer changes.

*   **Value Iteration:**
    *   **Process:** It directly computes $V^*$ by iteratively applying the **Bellman backup operator**, which takes the **max** reward possible at each state.
    *   **Intuition:** Each iteration $k$ represents the optimal value if the agent had a horizon of $k$ steps.
    *   **Convergence:** Guaranteed because the Bellman operator is a **contraction mapping** (with $\gamma < 1$), meaning each application of the operator reduces the distance between the current value function and the unique optimal value function.

### **Key Philosophical Distinctions**
While **Policy Iteration** maintains an explicit policy that acts for the full (infinite) horizon, it might be a poor policy early on. Conversely, **Value Iteration** always maintains an optimal value, but for the "wrong" (finite) horizon, building up to the infinite horizon through iteration.

Based on the source provided, here is a coding exercise designed to reinforce the core algorithms of **Tabular MDP Planning**: Policy Evaluation, Policy Iteration, and Value Iteration.

### **Coding Exercise: Tabular MDP Planning**

**Objective:** Implement the fundamental algorithms discussed in the lecture to find the optimal policy for a small tabular environment (similar to the 7-state Mars Rover example).

#### **The Environment Setup**
Assume we have an MDP defined by:
*   **States ($S$):** A set of 7 states.
*   **Actions ($A$):** Two actions, `A1` and `A2`.
*   **Dynamics ($P$):** A transition matrix where $P[s, a, s']$ is the probability of landing in state $s'$ given state $s$ and action $a$.
*   **Rewards ($R$):** $R[s, a]$ is the reward for taking action $a$ in state $s$.
*   **Gamma ($\gamma$):** The discount factor (e.g., 0.9).

In [ ]:

import numpy as np

class TabularMDP:
    def __init__(self, num_states, num_actions, gamma=0.9):
        self.S = num_states
        self.A = num_actions
        self.gamma = gamma
        # Transitions: P[state, action, next_state]
        self.P = np.zeros((self.S, self.A, self.S)) 
        # Rewards: R[state, action]
        self.R = np.zeros((self.S, self.A)) 

# Initialize a simple 3-state environment for testing
mdp = TabularMDP(num_states=3, num_actions=2)
# (In a real scenario, you would fill P and R with environment logic)

---

#### **Task 1: Iterative Policy Evaluation**
The lecture describes this as initializing values to zero and repeatedly applying the **Bellman backup** for a fixed policy until the value function stops changing.

**Instructions:** Complete the function to compute the value function $V^\pi$ for a given deterministic policy.


In [ ]:
def policy_evaluation(mdp, policy, threshold=1e-6):
    V = np.zeros(mdp.S)
    while True:
        delta = 0
        for s in range(mdp.S):
            v_old = V[s]
            # Use the deterministic policy to pick the action
            a = policy[s]
            
            # TODO: Implement the Bellman backup for a fixed policy
            # V[s] = R(s,a) + gamma * sum_{s'} (P(s'|s,a) * V[s'])
            new_v = mdp.R[s, a] + mdp.gamma * np.dot(mdp.P[s, a, :], V)
            
            V[s] = new_v
            delta = max(delta, abs(v_old - V[s]))
        
        if delta < threshold:
            break
    return V


---

#### **Task 2: Policy Iteration**
Policy iteration alternates between **evaluating** the current policy and **improving** it by choosing the action that maximizes the Q-function (the "argmax" step).

**Instructions:** Implement the policy improvement step.


In [ ]:
def policy_iteration(mdp):
    # Initialize a random deterministic policy
    policy = np.random.randint(0, mdp.A, mdp.S)

    while True:
        # 1. Evaluation
        V = policy_evaluation(mdp, policy)
        
        policy_stable = True
        for s in range(mdp.S):
            old_action = policy[s]
            
            # TODO: Policy Improvement Step
            # Calculate Q(s, a) for all actions and take the argmax
            q_values = []
            for a in range(mdp.A):
                q_sa = mdp.R[s, a] + mdp.gamma * np.dot(mdp.P[s, a, :], V)
                q_values.append(q_sa)
            
            policy[s] = np.argmax(q_values)
            
            if old_action != policy[s]:
                policy_stable = False
        
        # If policy doesn't change, it will never change again
        if policy_stable:
            break
            
    return policy, V


---

#### **Task 3: Value Iteration**
Unlike policy iteration, value iteration directly updates the value function by taking the **max** reward possible at each state (the optimal Bellman backup).

**Instructions:** Complete the loop to compute the optimal value function $V^*$.


In [ ]:
def value_iteration(mdp, threshold=1e-6):
    V = np.zeros(mdp.S) # Initialize values to zero
    while True:
        delta = 0
        for s in range(mdp.S):
            v_old = V[s]
            
            # TODO: Bellman Optimality Backup
            # V[s] = max_a [ R(s,a) + gamma * sum_{s'} (P(s'|s,a) * V[s']) ]
            q_values = [mdp.R[s, a] + mdp.gamma * np.dot(mdp.P[s, a, :], V) for a in range(mdp.A)]
            V[s] = max(q_values)
            
            delta = max(delta, abs(v_old - V[s]))
        
        # Check convergence using L-infinity norm
        if delta < threshold:
            break
    
    # Extract the final policy using argmax
    optimal_policy = np.zeros(mdp.S, dtype=int)
    for s in range(mdp.S):
        q_values = [mdp.R[s, a] + mdp.gamma * np.dot(mdp.P[s, a, :], V) for a in range(mdp.A)]
        optimal_policy[s] = np.argmax(q_values)
        
    return optimal_policy, V

### **Conceptual Review Questions (to answer after coding):**
1.  **Monotonicity:** Why is Policy Iteration guaranteed to improve the policy at every step, whereas Value Iteration is only guaranteed to move the Value Function closer to the unique optimal via contraction?
2.  **Termination:** Why can we guarantee that Policy Iteration will eventually stop in a tabular world?
3.  **Complexity:** If your state space $S$ was 1,000,000 states, why would you prefer these iterative methods over the analytical matrix inversion solution ($V = (I - \gamma P)^{-1} R$)?